In [ ]:
# 1. Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual styles for the plots
sns.set_style('whitegrid')
print("Libraries imported and styles set.")

In [ ]:
# 2. Load the raw dataset
df = pd.read_csv('./data/Rockfall.csv')
print("Dataset loaded successfully.")

In [ ]:
# 3. Remove leaky features that we identified earlier
# These features contain information about the outcome and would give a false sense of importance.
leaky_features = ['Susceptibility_Score', 'Impact_Level']
df_clean = df.drop(columns=leaky_features)
print(f"Removed leaky features: {leaky_features}")

In [ ]:
# 4. Identify the columns to analyze
# We separate them into numerical and categorical types for different plotting methods.
target_variable = 'Occurred'

numerical_features = df_clean.select_dtypes(include=np.number).columns.drop([target_variable, 'Latitude', 'Longitude', 'Month', 'Day'])
categorical_features = df_clean.select_dtypes(include='object').columns

print(f"\nFound {len(numerical_features)} numerical features to analyze.")
print(f"Found {len(categorical_features)} categorical features to analyze.")

In [ ]:
# --- 5. Analyze Numerical Features vs. Target ---
print("\n--- Generating plots for Numerical Features ---")
print("How to read these plots: Look for a clear difference in the SHAPE of the two violins (0 vs 1).")
print("A different shape means the feature has a different distribution when a rockfall occurs.")

for feature in numerical_features:
    plt.figure(figsize=(8, 5))
    sns.violinplot(x=target_variable, y=feature, data=df_clean)
    plt.title(f'{feature} vs. Rockfall Occurrence', fontsize=14)
    plt.xlabel('Rockfall Occurred (0 = No, 1 = Yes)', fontsize=10)
    plt.ylabel(feature, fontsize=10)

In [ ]:
# --- 6. Analyze Categorical Features vs. Target ---
print("\n--- Generating plots for Categorical Features ---")
print("How to read these plots: Look for a clear difference in the HEIGHT of the bars.")
print("A higher bar means a higher rate of rockfalls for that specific category.")

for feature in categorical_features:
    # Calculate the percentage of rockfalls for each category in the feature
    risk_df = df_clean.groupby(feature)[target_variable].value_counts(normalize=True).mul(100).rename('percent').reset_index()

    # Filter for only the 'Occurred' == 1 cases to plot the risk rate
    risk_df_filtered = risk_df[risk_df[target_variable] == 1]

    plt.figure(figsize=(10, 5))
    sns.barplot(x=feature, y='percent', data=risk_df_filtered)
    plt.title(f'Rockfall Rate by {feature}', fontsize=14)
    plt.xlabel(feature, fontsize=10)
    plt.ylabel('Rockfall Rate (%)', fontsize=10)
    plt.ylim(0, 100) # Keep the y-axis consistent for fair comparison
    plt.xticks(rotation=45) # Rotate labels if they are long
    plt.tight_layout()
    plt.show()

print("\n--- Deep Dive EDA Complete ---")